# Benchmark B: GPU 自己教師あり学習の較正(bench_gpu_ssl.ipynb)
**目的**: SSL 事前学習の **「画像/秒」を実測**し、
本番(10⁷ 枚 × 100 epoch)の GPU 時間と費用を外挿する。

## 実行環境(いつもと違うので注意)
- **インスタンス**: g5.xlarge(NVIDIA A10G ×1、4 vCPU)。オンデマンド約 $1.0/h、スポット約 $0.3〜0.5/h
- **AMI**: 「**Deep Learning OSS Nvidia Driver AMI GPU PyTorch (Amazon Linux 2023)**」を選ぶこと。
  NVIDIA ドライバと PyTorch が設定済みで、素の AL2023 だとドライバ設置から必要になる
- スポットで立てる場合: 起動画面の「高度な詳細」→ 購入オプションで「スポットインスタンス」にチェック
- ロール(roman-ec2-role)とセッションマネージャー接続はいつも通り。追加 pip: `pip install s3fs`
- **Service Quotas の G 系枠(vCPU ≥ 4)が承認済みであること**
- 所要 1〜2 時間、費用 $1〜2 の見込み。終わったら必ず停止(できれば終了)

In [ ]:
MY_BUCKET = "YOUR-BUCKET-roman-scratch"
CUTOUT_SIZE = 64
BATCH = 512
N_STEPS = 300           # 計測ステップ数(ウォームアップ後)
GPU_USD_ON_DEMAND = 1.006
GPU_USD_SPOT = None     # 起動時に表示された実際のスポット価格を記入

In [ ]:
import io, time, numpy as np, s3fs, torch, torch.nn as nn
import torchvision
print(torch.__version__, "CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

# Benchmark A の切り出しを学習素材として読み込む
fs = s3fs.S3FileSystem()
files = fs.glob(f"{MY_BUCKET}/bench_cutouts/*.npz")
arrs = []
for p in files:
    with fs.open(p, "rb") as f:
        a = np.load(io.BytesIO(f.read()))["cutouts"]
    if a.ndim == 3:
        arrs.append(a)
data = np.concatenate(arrs)
data = np.nan_to_num(data)
# 簡易正規化(asinh ストレッチ)
data = np.arcsinh(data / (np.nanstd(data) + 1e-8)).astype(np.float32)
print(data.shape, "cutouts loaded")

In [ ]:
# SimCLR 風の最小構成: ResNet18 + 射影ヘッド + NT-Xent
device = "cuda"
enc = torchvision.models.resnet18(num_classes=128)
enc.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)   # 1 チャンネル入力に変更
model = enc.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

def augment(x):
    # 反転・90 度回転・軽いノイズ(天文画像向けの最小限)
    if torch.rand(1) < 0.5: x = torch.flip(x, [-1])
    k = int(torch.randint(0, 4, (1,)))
    x = torch.rot90(x, k, [-2, -1])
    return x + 0.05 * torch.randn_like(x)

def nt_xent(z1, z2, tau=0.2):
    z = torch.cat([z1, z2]); z = nn.functional.normalize(z, dim=1)
    sim = z @ z.T / tau
    n = z1.shape[0]
    sim.fill_diagonal_(-1e9)
    targets = torch.cat([torch.arange(n, 2*n), torch.arange(0, n)]).to(device)
    return nn.functional.cross_entropy(sim, targets)

X = torch.from_numpy(data).unsqueeze(1)   # (N,1,64,64)
print("tensor:", X.shape)

In [ ]:
# ウォームアップ 20 step → N_STEPS 計測
model.train()
def step():
    idx = torch.randint(0, X.shape[0], (BATCH,))
    xb = X[idx].to(device, non_blocking=True)
    z1, z2 = model(augment(xb)), model(augment(xb))
    loss = nt_xent(z1, z2)
    opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

for _ in range(20): step()
torch.cuda.synchronize()
t0 = time.perf_counter()
losses = [step() for _ in range(N_STEPS)]
torch.cuda.synchronize()
dt = time.perf_counter() - t0

imgs_per_s = N_STEPS * BATCH * 2 / dt     # 2 view なので ×2
print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"throughput: {imgs_per_s:,.0f} images/s (batch {BATCH}, 64px, A10G)")

In [ ]:
# --- 本番規模への外挿 ---
N_TARGET, EPOCHS, N_RUNS = 1e7, 100, 8
gpu_h_per_run = N_TARGET * EPOCHS * 2 / imgs_per_s / 3600
total_h = gpu_h_per_run * N_RUNS
print(f"1 run (10⁷枚×{EPOCHS}ep): {gpu_h_per_run:,.0f} GPU-h")
print(f"{N_RUNS} runs 合計      : {total_h:,.0f} GPU-h")
print(f"オンデマンド           : ${total_h*GPU_USD_ON_DEMAND:,.0f}")
if GPU_USD_SPOT:
    print(f"スポット               : ${total_h*GPU_USD_SPOT:,.0f}")
print("(推論/異常検知は forward のみ ≈ 学習の数%なので丸め込みで十分)")

**終わったら**: g5.xlarge は必ず**停止 or 終了**(高いので放置注意)。
`bench_cutouts/` はここで消して OK。
おまけ: この較正がそのまま「本番 SSL パイプラインの雛形」になっている
(データ読み込み・増強・損失は本番と同じ構成)。